# SOLUCIÓN PROBLEMA 1 SIMULACRO - FINAL Probabilidad y Estadística - iMAT-ICAI - 18 de mayo 2024

# Vale el 0% de la nota del examen
---

NO CAMBIES EL NOMBRE AL NOTEBOOK, SIMPLEMENTE ESCRIBE AQUÍ TUS DATOS
### Nombre: RELLENAR
### Apellidos: RELLENAR
---

### Se trata de un ejercicio sencillo pero bastante completo que permite verificar que puedes trabajar sin problemas con SEB.

Responde las preguntas utilizando instrucciones print() para mostrar claramente tu respuesta en la ejecución de la celda, explicando cómo lo resuelves. Utiliza comentarios en tu código para facilitar su lectura.

### Importamos todos los paquetes aquí por comodidad

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
from scipy import optimize as opt  # para estimación por máxima verosimilitud

from statsmodels.nonparametric.kde import KDEUnivariate

import statsmodels.api as sm

from anova import anova1, anova2

### a) Crea una v.a. X que siga una distribución triangular con parámetros a = 2, b = 10 y c = 20. Crea también una v.a. Y que siga una distribución de Poisson de parámetro lmda = 10

In [ ]:
# Respuesta

a = 2 # min
c = 10 # moda 
b = 20 # max
loc = a
escalado = b-a
forma = (c-a)/escalado
X = stats.triang(loc=loc, scale=escalado, c = forma)

# crea la v.a. "Y" que sigue una Poisson
lmda= 10
Y = stats.poisson(lmda)

### b) Representa en la misma gráfica las funciones de masa de probabilidad y de densidad de las variables aleatorias.

In [ ]:
# Respuesta

# puntos en los que se evalua las funciones para representarlas
xsX = np.linspace(X.ppf(0.0001)-5, X.ppf(0.9999)+5, 10000)
xsY = range(0, 25)

# pinta sincronizadamente ambas funciones
fig = plt.figure(1, figsize=(16, 4))

plt.fill_between( xsX, X.pdf(xsX), color="orange", alpha=0.3)
plt.plot(xsX, X.pdf(xsX), color="orange", label = 'pdf(X)')

plt.stem(xsY, Y.pmf(xsY), basefmt='none', label = 'pmf(Y)')

plt.legend()
plt.grid()
plt.show()

### c) Genera 10000 números aleatorios de la v.a. X y guárdalos en la variable "x". Haz lo mismo con la v.a. Y y lo guardas en la variable "y". Fija la semilla antes de hacerlo en el valor 999. Representa el histograma de ambos conjuntos de datos alteatorios.

In [ ]:
# Respuesta

semilla = 999
np.random.seed(semilla)

# genera n números aleatorios segun VAs
n = 10000
x = X.rvs(size = n)
y = Y.rvs(size = n)

# pinta sincronizadamente ambas funciones
fig = plt.figure(1, figsize=(16, 4))

plt.hist(x, bins = 40, color="orange", alpha = 0.4, label = 'pdf(X)')
plt.hist(y, bins = 90, color="blue", alpha = 0.4, label = 'pmf(Y)')

plt.legend()
plt.grid()
plt.show()

### d) Calcula la matriz de covarianzas de las variables x e y. Representa el gráfico de nube de puntos incluyendo los histogramas marginales. ¿Están relacionadas ambas variables?

In [ ]:
# Respuesta

# matriz de covarianzas
mat_cov = np.cov(x,y)
print('Matriz covarianzas:')
print(np.round(mat_cov, 3)) # Redondeando a solo tres decimales para verlo mejor

# completando el scatter con los histogramas marginales
fig = plt.figure(2)
sns.jointplot(x=x, y=y)
plt.show()

print('Obviamente no están relacionadas, se han generado de manera independiente.')

### e) Calcula un intervalo de confianza al 99% para las medias de las v.a. X e Y utilizando Bootstrap con 10000 submuestras. Representa en la misma gráfica dichos intervalos. ¿Existen diferencias significativas entre ambas medias?

In [ ]:
# Respuesta

# params para llamar a bootstrap (método BCa, el que se suele usar normalmente)
muestra_orig = (x,)  # tiene que ser una secuencia
R = 10000
estadistico = np.mean
nivel_confianza = 0.99

res_BCa = stats.bootstrap(muestra_orig, estadistico, confidence_level = nivel_confianza,  n_resamples = R, method = 'BCA', random_state = 999)

# devuelve como resultado el IC y el SE del estadístico
lim_inf_X, lim_sup_X = res_BCa.confidence_interval
SE_BCa_X = res_BCa.standard_error

print(f'IC {100*nivel_confianza}% estimado según método BCa para la media: [{lim_inf_X}, {lim_sup_X}]')
print(f'Error estándar del estimador media muestral, estimado según método BCa: {SE_BCa_X}')

# params para llamar a bootstrap (método BCa, el que se suele usar normalmente)
muestra_orig = (y,)  # tiene que ser una secuencia
R = 10000
estadistico = np.mean
nivel_confianza = 0.99

res_BCa = stats.bootstrap(muestra_orig, estadistico, confidence_level = nivel_confianza,  n_resamples = R, method = 'BCA', random_state = 999)

# devuelve como resultado el IC y el SE del estadístico
lim_inf_Y, lim_sup_Y = res_BCa.confidence_interval
SE_BCa_Y = res_BCa.standard_error

print(f'IC {100*nivel_confianza}% estimado según método BCa para la media: [{lim_inf_Y}, {lim_sup_Y}]')
print(f'Error estándar del estimador media muestral, estimado según método BCa: {SE_BCa_Y}')

print('\nHay diferencias significativas entre las medias porque los IC creados al 99% no solapan.')

In [ ]:
# gráficamente

x_media = np.mean(x)
y_media = np.mean(y)

fig = plt.figure(1, figsize=(16, 6))
plt.errorbar(x_media, 1, xerr = np.array([[x_media - lim_inf_X], 
            [lim_sup_X - x_media]]), fmt = 'o', label = f'IC [{lim_inf_X}, {lim_sup_X}] al {nivel_confianza*100}%')
plt.errorbar(y_media, 2, xerr = np.array([[y_media - lim_inf_Y], 
            [lim_sup_Y - y_media]]), fmt = 'o', label = f'IC [{lim_inf_Y}, {lim_sup_Y}] al {nivel_confianza*100}%')

plt.legend()
plt.show()

### f) Calcula la probabilidad de que, al mismo tiempo, la v.a. X<10 y que la v.a. Y>=5.

In [ ]:
# Respuesta

print('Se pide P(X<10 & Y>=5).')
prob_x_menor_10 = X.cdf(10)
prob_y_mayor_igual_5 = 1 - Y.cdf(4)

prob = prob_x_menor_10 * prob_y_mayor_igual_5
print(F'Como las v.a. son independientes la P(X<10 & Y>=5) = P(X<10) * P(Y>=5) =  P(X<10) * (1-P(Y<=4)) = {prob}')

### g) Se quiere modelar la nueva v.a. Z = X + Y. Calcula el valor esperado y la varianza de la variable Z.

In [ ]:
# Respuesta

E_Z = X.mean() + Y.mean()
print(f'El valor E[Z] = E[X] + E[Y] = { X.mean()} + {Y.mean()} = {E_Z}')

var_Z = X.var() + Y.var()
print(f'Como X e Y son independientes, el valor Var[Z] = Var[X] + Var[Y] = { X.var()} + {Y.var()} = {var_Z}')

### h) Determina la distribución de Z con un enfoque empírico a partir de la muestra de datos generada mediante simulación de (x,y). Calcula la media y varianza de los datos simulados de Z y compara con el cálculo anterior. 

In [ ]:
# Respuesta

print('A partir de los datos simulados de (x,y) obtenemos z sumando directamente.')
z = x + y

z_med = np.mean(z)
z_var = np.var(z)
print(f'La media de z es {z_med} y su varianza {z_var}, muy parecido a lo calculado teóricamente.')

### i) Representa en la misma gráfica los histogramas de las variables simuladas x, y, z. Elige un nº de bins apropiado para cada variable para que se observen bien las distribuciones. ¿A qué distribución recuerda el histograma de z?

In [ ]:
# histograma
fig = plt.figure(1, figsize=(16, 4))

plt.hist(x, bins = 40, color="orange", alpha = 0.4, label = 'x')
plt.hist(y, bins = 90, color="blue", alpha = 0.4, label = 'y')
plt.hist(z, bins = 40, color="red", alpha = 0.4, label = 'z')

plt.legend()
plt.grid()
plt.show()

print('La distribución de z se quiere parecer a una normal.')

### j) Realiza un contraste no paramétrico para determinar si la distribución de z puede ser una normal. Razona con un nivel de significación del 1%. Completa el análisis con un gráfico qqplot y concluye claramente.

In [ ]:
# Respuesta

# nombre de la distribución teórica que se contrasta (norm, uniform, triang, expon, ...)
distribucion_teo = stats.norm

# genera qqplot de la variable x
sm.qqplot(z, line='45', fit=True, dist=distribucion_teo)
plt.title('H0: Se distribuye como una normal')
plt.grid(); plt.show()

# nivel de significación
alfa = 0.01

# realiza el test
ks = sm.stats.diagnostic.kstest_normal(z, dist='norm', pvalmethod='table')

max_d   = ks[0]
p_valor = ks[1]
print(f'Contraste KS es una normal:: máxima distancia observada = {max_d} -> p_valor = {p_valor}')

# concluye
print(f'Como el pvalor < {alfa}, se puede rechazar H0 con un nivel de significación del {alfa*100}% ya que la distancia máxima observada es suficientemente grande.')
print(f'Según el test no paramétrico KS se puede afirmar, con un nivel de significación del {alfa*100}%, que los datos de la muestra simulada de Z no siguen una distribución normal.')
print('En el gráfico qqplot se puede observar que la cola de la izquierda se aleja bastante')

In [ ]:
# Representación visual del contraste

# estima coefs de la normal
va_teo = stats.norm(np.mean(z), np.std(z))

# calcula acumulada empírica
ecdf = sm.distributions.empirical_distribution.ECDF(z)

# crea gráfica
xs = np.linspace(min(va_teo.ppf(0.01), min(z)), max(va_teo.ppf(1-0.01), max(z)), 1000)

fig = plt.figure(1, figsize=(12, 4))
plt.step(ecdf.x, ecdf.y, label=f'ecdf (n={n})', where='post')
plt.plot(xs, va_teo.cdf(xs), label=f'Normal ($mu$={np.mean(z)}, $\sigma$={np.std(z)})')
sns.rugplot(z, height = 0.025, lw = 0.4, color = 'red')
plt.title(f'KS: max D = {max_d}, p-valor = {p_valor}')
plt.grid()
plt.xlabel('z');plt.ylabel('cdf');plt.legend()
plt.show()

print('Puede sorprender que se rechace la normalidad de z, la explicación está en el tamaño muestral, muy grande, que '
      'permite ser mucho más exigente.')

### k) Ajusta por máxima verosimilitud una normal a la muestra z. Crea la v.a. Correspondiente y representa su función de densidad junto con el histograma de la muestra z.

In [ ]:
# Respuesta

# definimos la función de verosimilitud logarítmica para maximizarla
    
def verosimilitud_log(parameters):
    # sacamos los parámetros
    param1, param2 = parameters
    
    # La función de verosimilitud logarítmica depende del problema y requiere
    # conocer la distribución de la v.a. bajo estudio, que dependerá de los params
    # a estimar por MV
    
    # Ejemplo estimando la media y desv. típica de una normal:
    logLikelihood = np.sum(stats.norm.logpdf(z, loc=param1, scale=param2))  # <--- logpdf es el log de la pdf, ya está preparado ...
    
    # Como la función minimiza y queremos los params que maximizan la verosimilitud
    # simplemente la multimplicamos por -1: max(loglike) = min(-loglike)
    neg_logLikelihood = -1 * logLikelihood
    return neg_logLikelihood 


In [ ]:
# Maximizamos la función de verosimilitud logarítmica (minimizando -1 * funcion)
mle_model = opt.minimize(verosimilitud_log, np.array([1,1]), method='L-BFGS-B')

# cogemos los coefs estimados
param1 = mle_model.x[0]
param2 = mle_model.x[1]

print(mle_model)
print('Parámetros estimados por MV:\n'
      f'  - parámetro 1: {param1}\n'
      f'  - parámetro 2: {param2}')

print('\nLógicamente los parámetros estimados coinciden con la media y la desviación típica de la muestra de Z.')

# creamos la v.a. Z y representamos
Z = stats.norm(loc = param1, scale = param2)

xs = np.linspace(Z.ppf(0.00001), Z.ppf(1-0.00001), 1000)

plt.figure(1, figsize=(15, 5))
plt.hist(z, 40, density = True, alpha = 0.4, label =f'Histograma')
plt.fill_between(xs, Z.pdf(xs), color="orange", alpha=0.3)
plt.plot(xs, Z.pdf(xs), color="orange", label = 'Normal estimada')
sns.rugplot(z, height = 0.025, lw = 0.4, color = 'red', label = 'muestra z')
plt.title(f'Normal estimada por máx. verosimilitud')
plt.grid(); plt.legend()
plt.show()

### l) Ajusta un KDE a la muestra de z y representa en la misma gráfica el histograma de z, el KDE estimado y la normal estimada previamente. Utiliza una función kernel gaussiana y estima automáticamente el ancho del banda del KDE.

In [ ]:
# Respuesta

n = len(z)

# estima el KDE

# fija la función kernel a utilizar

kernel_fun = 'gau'

# fija el bandwidth (si auto lo estima la función de ajuste)
tipo_bw = 'auto' # 'manual'

if tipo_bw == 'auto':
    bandwidth = 'normal_reference' # otras opciones: bandwidth = 'scott', bandwidth = 'silverman'
else:
    bandwidth = 0.6 # fija bandwidth exacto

    
# estima kde
kde = KDEUnivariate(z)
kde.fit(kernel = kernel_fun, bw = bandwidth, fft = True)

# calcula verosimilitud log media de ambas aproximaciones
logLikelihood_med_kde = np.sum(np.log(kde.evaluate(z))) / n

print(f'Verosimilitud logarítmica media de KDE: {logLikelihood_med_kde}')

# fija nbins para mostrar histograma
nbins_hist = 40

plt.figure(1, figsize=(15, 5))
plt.hist(z, nbins_hist, density = True, alpha = 0.4, label =f'Histograma nbins={nbins_hist}')
plt.plot(kde.support, kde.density, color = 'black', linewidth = 3, label =f'pdf KDE bw={kde.bw}')

# densidad de la normal
xs = np.linspace(Z.ppf(0.00001), Z.ppf(1-0.00001), 1000)
plt.fill_between(xs, Z.pdf(xs), color="orange", alpha=0.3)
plt.plot(xs, Z.pdf(xs), color="orange", label = 'Normal estimada')

sns.rugplot(z, height = 0.025, lw = 0.4, color = 'red')
plt.title(f'KDE con bandwidth = {kde.bw} | bw fijado con método: {kde.bw_method}')
plt.grid(); plt.legend()
plt.show()

### m) A partir de la muestra creada con las variables (y,z), realiza un ANOVA para estudiar si z (VAR_Z) depende del factor (VAR_Y). Concluye claramente y representa los coeficientes del modelo de medias asociado.

In [ ]:
# Respuesta

df =  pd.DataFrame(list(zip(y, z)), columns=['VAR_Y', 'VAR_Z'])
# muestra df
df

salida = 'VAR_Z'
factor = 'VAR_Y'
formula = f'{salida} ~ C({factor})'
print('Fórmula: ', formula)

# calcula ANOVA
tabla_anova, modelo_medias = anova1(formula, df)

# muestra tabla anova
print('Tabla ANOVA:')
print(tabla_anova)
print('\n')
print('Grados de libertad totales: ' , tabla_anova.df.sum())

print('\nComo el pvalor es nulo, se puede rechazar la H0 (todos los coefs asociados al factor VAR_Y son iguales), '
      'a favor de la H1 (hay diferencias en VAR_Z dependiendo del valor del factor VAR_Y).')

In [ ]:
# muestra modelo de medias
print('Modelo de medias:')

# gráficamente
fig = plt.figure(1, figsize=(15, 4))
plt.subplot(1,2,1)
plt.bar(modelo_medias.index[0], modelo_medias.Mean.iloc[0])
plt.ylabel('Media global')

plt.subplot(1,2,2)
plt.bar(modelo_medias.index[1:], modelo_medias.Mean.iloc[1:])
plt.ylabel('Coeficientes alfa')
plt.xlabel(factor)
plt.show()